# 04 - Modeling

Six models so the metrics are interpretable:

- **Persistence** `y(t+1)=y(t)` and **Seasonal-naive** `y(t+1)=y(t+1-24)` - baselines that make RMSE meaningful.
- **Ridge** - transparent linear reference.
- **LightGBM / XGBoost** - gradient-boosted trees with early stopping on a time-ordered validation fold + depth and L1/L2 regularisation.
- **LSTM** (PyTorch) - a sequential neural net on 24 h windows, dropout + early stopping.

**Overfitting defence:** chronological split, leak-safe features, scalers fit on train only, early stopping, regularisation, dropout.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import config as C, features as fe, models as M
from src.evaluate import regression_metrics, skill_vs_baseline
cleaned = pd.read_csv(C.DATA_PROCESSED, index_col=0, parse_dates=True)
feat = fe.build_features(cleaned)
(Xtr,ytr),(Xva,yva),(Xte,yte) = fe.time_split(feat)
ts = cleaned[C.TARGET]
res = {}
res['Persistence']  = regression_metrics(yte, M.persistence_forecast(yte, ts))
res['SeasonalNaive']= regression_metrics(yte, M.seasonal_naive_forecast(yte, ts))
res['Ridge']    = regression_metrics(yte, M.fit_ridge(Xtr,ytr).predict(Xte))
lgbm = M.fit_lightgbm(Xtr,ytr,Xva,yva)
res['LightGBM'] = regression_metrics(yte, lgbm.predict(Xte))
xgbm = M.fit_xgboost(Xtr,ytr,Xva,yva)
res['XGBoost']  = regression_metrics(yte, xgbm.predict(Xte))
rmse0 = res['Persistence']['RMSE']
for k,v in res.items(): v['skill%']=round(skill_vs_baseline(v['RMSE'],rmse0),2)
import pandas as pd
pd.DataFrame(res).T.sort_values('RMSE')[['RMSE','MAE','R2','skill%']].round(3)

,RMSE,MAE,R2,skill%
XGBoost,2.235,1.578,0.807,19.88
LightGBM,2.274,1.630,0.801,18.48
Ridge,2.723,2.079,0.714,2.37
Persistence,2.790,1.769,0.700,0.00
SeasonalNaive,4.839,3.357,0.098,-73.46


LightGBM and XGBoost beat persistence by ~18-20% RMSE; the seasonal-naive baseline is *worse* than persistence because at a 1-hour horizon the most recent reading is far more informative than the same hour yesterday. The full run (`run_pipeline.py`) also trains the LSTM.